# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate the record sets, fields, and columns using their `@id` fields.

In [ ]:
# List all record sets using their @id
record_sets = dataset.metadata.record_sets
print('RecordSets:')
for rs in record_sets:
    print(f"  @id: {rs['@id']} | name: {rs['name']}")

# List fields and columns for each record set
for rs in record_sets:
    print(f"\nFields for RecordSet {rs['@id']}:")
    for field in rs['fields']:
        print(f"    Field @id: {field['@id']} | name: {field['name']}")
        if 'column' in field:
            # Columns may be a list or a single dictionary
            cols = field['column']
            if isinstance(cols, dict):
                cols = [cols]
            for col in cols:
                print(f"      Column @id: {col['@id']} | name: {col['name']}")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis.

Use the record set and field `@id`s from the overview above.

This section demonstrates extracting all record sets.

In [ ]:
# Get the list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet {record_set_id} columns: {df.columns.tolist()}")
        print(f"First five rows of {record_set_id}:")
        display(df.head())
    else:
        print(f"No records found for RecordSet {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, and grouping data.

We'll select a numeric field for filtering and normalization, using the field's `@id` where possible.

In [ ]:
# Select a record set and numeric field for demonstration
# The dataset has one main recordset containing clinical information
main_rs_id = record_set_ids[0] if record_set_ids else None

if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    print(f"Columns in recordset {main_rs_id}: {df.columns.tolist()}")

    # Try to select 'Age' as a numeric field (field @id likely something like 'https://api.app.sen.science/frontiers/7862866/age')
    numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or 'Age' in col]
    numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]

    print(f"Selected numeric field: {numeric_field}")

    threshold = 50
    # Filter records with Age > threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field, e.g., Sex
    group_field_candidates = [col for col in df.columns if 'sex' in col.lower() or 'Sex' in col]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df)
else:
    print("No main record set found or no dataframe extracted.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

For example, plot the age histogram and age distributions grouped by sex.

In [ ]:
# Plot histogram and boxplot for the numeric field
if main_rs_id and main_rs_id in dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
The FAIR² dataset package was successfully loaded and explored using the `mlcroissant` library.

- Data extraction is achieved using `@id` for record sets, fields, and columns.
- We performed basic EDA, including filtering, normalization, and grouping by demographic fields.
- Visualizations provided further insights into clinical variable distributions.

This workflow can be adapted for deeper analysis or integration into ML pipelines.